## Percentage of tabular and image features selected

In [8]:
import os
import pandas as pd
import numpy as np

BASE_PATH = "../logs/MedViT2-nopt/img_raw_features/exp3_mlp"
BEST_REL_PATH = os.path.join("edca", "edca_fold1", "best_data.csv")

# full original train dataset, used to get total available features
FULL_DATA_PATH = "../data/datasets/MedViT2-nopt/all_data/retrospective/retrospective_fusion_img_emb_plus_raw_features.csv"

IMG_PREFIX = "img_emb"
DROP_COLS = {"processo"}
LABEL_COLS = {"class", "target_class"}


def is_valid_feature(col):
    c = str(col).strip()
    c_low = c.lower()

    if c == "":
        return False
    if c_low.startswith("unnamed"):
        return False
    if c_low in LABEL_COLS:
        return False
    if c_low in DROP_COLS:
        return False
    return True


# total available features in the original dataset
df_full = pd.read_csv(FULL_DATA_PATH)
full_feature_cols = [c for c in df_full.columns if is_valid_feature(c)]

total_img_features = sum(str(c).startswith(IMG_PREFIX) for c in full_feature_cols)
total_tab_features = len(full_feature_cols) - total_img_features
total_features_available = len(full_feature_cols)

print(f"Total available features: {total_features_available}")
print(f"  Total tabular features: {total_tab_features}")
print(f"  Total image features:   {total_img_features}")

# selected features per run
runs = sorted([d for d in os.listdir(BASE_PATH) if d.startswith("exp_")])
print(f"Found {len(runs)} runs")

rows = []

for run in runs:
    best_path = os.path.join(BASE_PATH, run, BEST_REL_PATH)

    if not os.path.exists(best_path):
        print(f"[WARN] missing: {best_path}")
        continue

    df = pd.read_csv(best_path)
    feature_cols = [c for c in df.columns if is_valid_feature(c)]

    selected_total = len(feature_cols)
    selected_img = sum(str(c).startswith(IMG_PREFIX) for c in feature_cols)
    selected_tab = selected_total - selected_img

    # composition inside selected set
    tab_pct_selected_set = 100 * selected_tab / selected_total if selected_total else 0
    img_pct_selected_set = 100 * selected_img / selected_total if selected_total else 0

    # coverage relative to all available features of each type
    tab_pct_of_all_tab = 100 * selected_tab / total_tab_features if total_tab_features else 0
    img_pct_of_all_img = 100 * selected_img / total_img_features if total_img_features else 0

    # overall coverage relative to all available features
    total_pct_of_all_features = 100 * selected_total / total_features_available if total_features_available else 0

    rows.append({
        "run": run,
        "selected_total_features": selected_total,
        "selected_tab_count": selected_tab,
        "selected_img_count": selected_img,
        "tab_%_within_selected": round(tab_pct_selected_set, 3),
        "img_%_within_selected": round(img_pct_selected_set, 3),
        "tab_%_of_all_tab_features": round(tab_pct_of_all_tab, 3),
        "img_%_of_all_img_features": round(img_pct_of_all_img, 3),
        "total_%_of_all_features": round(total_pct_of_all_features, 3),
    })

results = pd.DataFrame(rows).sort_values("run")

print("\nPer-run percentages:")
print(results)

summary = pd.DataFrame([{
    "run": "MEAN",
    "selected_total_features": round(results["selected_total_features"].mean(), 3),
    "selected_tab_count": round(results["selected_tab_count"].mean(), 3),
    "selected_img_count": round(results["selected_img_count"].mean(), 3),
    "tab_%_within_selected": round(results["tab_%_within_selected"].mean(), 3),
    "img_%_within_selected": round(results["img_%_within_selected"].mean(), 3),
    "tab_%_of_all_tab_features": round(results["tab_%_of_all_tab_features"].mean(), 3),
    "img_%_of_all_img_features": round(results["img_%_of_all_img_features"].mean(), 3),
    "total_%_of_all_features": round(results["total_%_of_all_features"].mean(), 3),
}, {
    "run": "STD",
    "selected_total_features": round(results["selected_total_features"].std(), 3),
    "selected_tab_count": round(results["selected_tab_count"].std(), 3),
    "selected_img_count": round(results["selected_img_count"].std(), 3),
    "tab_%_within_selected": round(results["tab_%_within_selected"].std(), 3),
    "img_%_within_selected": round(results["img_%_within_selected"].std(), 3),
    "tab_%_of_all_tab_features": round(results["tab_%_of_all_tab_features"].std(), 3),
    "img_%_of_all_img_features": round(results["img_%_of_all_img_features"].std(), 3),
    "total_%_of_all_features": round(results["total_%_of_all_features"].std(), 3),
}])

final = pd.concat([results, summary], ignore_index=True)

out_csv = "../results/img_raw_features/exp3_mlp/embedding_type_percentages_per_run.csv"
final.to_csv(out_csv, index=False)

print("\nSaved →", out_csv)
print("\nSummary:")
print(summary)

Total available features: 860
  Total tabular features: 92
  Total image features:   768
Found 30 runs

Per-run percentages:
                               run  selected_total_features  \
0   exp_2026-02-22 10:50:49.169392                      320   
1   exp_2026-02-22 12:21:16.817206                      199   
2   exp_2026-02-22 14:13:21.551532                      860   
3   exp_2026-02-22 15:43:02.651153                      860   
4   exp_2026-02-22 16:23:42.218032                      508   
5   exp_2026-02-22 17:02:18.904715                      860   
6   exp_2026-02-22 19:27:02.673682                      680   
7   exp_2026-02-22 21:09:34.992209                      860   
8   exp_2026-02-22 22:40:12.031767                      765   
9   exp_2026-02-22 23:52:21.955909                      860   
10  exp_2026-02-23 00:58:53.968026                      860   
11  exp_2026-02-23 02:09:22.054624                       36   
12  exp_2026-02-23 03:21:01.565287                      

## Count the number of times that each tabular feature is selected in best individual

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

BASE_PATH = "../logs/MedViT2-nopt/3planes_raw_features/exp3_mlp"  

def is_tabular_feature(col):
    """Return True if column is tabular (not image embedding / class / unnamed)."""
    #if col.lower().startswith("img_emb"): # mudar consoante o dataset que estamos a analisar
    if col.lower().startswith("image"):
        return False
    if col.lower() == "target_class":
        return False
    if col.lower().startswith("unnamed"):
        return False
    if col.strip() == "":
        return False
    return True


feature_counter = Counter()
run_feature_sets = {}

runs = sorted([d for d in os.listdir(BASE_PATH) if d.startswith("exp_")])

print(f"Found {len(runs)} runs")

for run in runs:
    best_path = os.path.join(BASE_PATH, run, "edca", "edca_fold1","best_data.csv")

    if not os.path.exists(best_path):
        print("Missing:", best_path)
        continue

    df = pd.read_csv(best_path)

    tabular_cols = [c for c in df.columns if is_tabular_feature(c)]

    # store features per run
    run_feature_sets[run] = set(tabular_cols)

    # count feature presence once per run
    feature_counter.update(tabular_cols)


counts_df = (
    pd.DataFrame(
        [{"feature": f, "count": c} for f, c in feature_counter.items()]
    )
    .sort_values("count", ascending=False)
)

print("\nTop features:")
print(counts_df.head(20))

counts_df.to_csv("../results/3planes_raw_features/exp3_mlp/tabular_feature_usage_counts.csv", index=False)
print("\nSaved → tabular_feature_usage_counts.csv")

max_count = counts_df["count"].max()
top_features = counts_df[counts_df["count"] == max_count]["feature"].tolist()

print("\nMost frequent features:", top_features)
print("Max count:", max_count)

print("\nRuns where top features are missing:")

for feature in top_features:
    missing_runs = []

    for run in runs:
        best_path = os.path.join(BASE_PATH, run, "edca", "edca_fold1", "best_data.csv")
        if not os.path.exists(best_path):
            continue

        df = pd.read_csv(best_path)

        if feature not in df.columns:
            missing_runs.append(run)
        
        num_runs = len(run_feature_sets)

        # features that appear in (num_runs - 1) runs → e.g., 29/30
        almost_all_features = [
            f for f, c in feature_counter.items()
            if c == num_runs - 1
        ]

print(f"\nFeatures appearing in {num_runs-1}/{num_runs} runs:")
for f in almost_all_features:
    print(" ", f)

    print(f"\nFeature: {feature}")
    if missing_runs:
        print("Missing in runs:")
        for r in missing_runs:
            print("  ", r)
    else:
        print("Appears in all runs")

print("\nRuns where these features are missing:")

for run, feats in run_feature_sets.items():
    missing = [f for f in almost_all_features if f not in feats]

    if missing:
        print(f"\n{run} is missing:")
        for f in missing:
            print("  ", f)



Found 30 runs

Top features:
                           feature  count
3                            Idade     29
2               PartoTermoAnterior     29
10              PartoAntTempoMeses     29
17                  EPF(percentil)     29
11                              IG     28
21              PartoAntTipo_1-csa     28
13                    EPF(pesoemg)     28
7                         IMCfinal     28
8                  AumentoPonderal     27
1                            Gesta     27
0                         Paridade     27
6                           Altura     27
22    PartoAntTipo_2-instrumentado     27
23               PartoAntTipo_3-PE     27
14                 Eco3ºT(semanas)     27
28  MotivoCSAant_10-EFNTintraparto     27
72                          PC(mm)     27
71                         DPB(mm)     27
20                  MetodoInd_4-SF     26
5                        PesoFinal     26

Saved → tabular_feature_usage_counts.csv

Most frequent features: ['Idade', 'PartoTermoA

## Percentage of tabular and image features selected in 3planes + features tabulares

In [11]:
import os
import pandas as pd
import numpy as np

BASE_PATH = "../logs/MedViT2-nopt/3planes_raw_features/exp3_mlp"
BEST_DATA_REL = os.path.join("edca", "edca_fold1", "best_data.csv")

FULL_DATA_PATH = "../data/datasets/MedViT2-nopt/all_data/retrospective/retrospective_femur_head_abdomen_emb_plus_raw_features.csv"

PREFIXES = ["image_femur", "image_head", "image_abdomen"]
NON_FEATURE_COLS = {"class", "target_class", "processo"}


def is_valid_feature(col):
    c = str(col).strip()
    c_low = c.lower()

    if c == "":
        return False
    if c_low.startswith("unnamed"):
        return False
    if c_low in NON_FEATURE_COLS:
        return False
    return True


# totals from full dataset
df_full = pd.read_csv(FULL_DATA_PATH)
full_feature_cols = [c for c in df_full.columns if is_valid_feature(c)]

total_available = len(full_feature_cols)
total_femur = sum(str(c).startswith("image_femur") for c in full_feature_cols)
total_head = sum(str(c).startswith("image_head") for c in full_feature_cols)
total_abdomen = sum(str(c).startswith("image_abdomen") for c in full_feature_cols)
total_image = total_femur + total_head + total_abdomen
total_tabular = total_available - total_image

print(f"Total available features: {total_available}")
print(f"  Femur:   {total_femur}")
print(f"  Head:    {total_head}")
print(f"  Abdomen: {total_abdomen}")
print(f"  Image total: {total_image}")
print(f"  Tabular: {total_tabular}")

runs = sorted([d for d in os.listdir(BASE_PATH) if d.startswith("exp_")])
rows = []
print(f"Found {len(runs)} experiments in: {BASE_PATH}")

for run in runs:
    best_path = os.path.join(BASE_PATH, run, BEST_DATA_REL)

    if not os.path.exists(best_path):
        print(f"[WARN] Missing best_data.csv: {best_path}")
        continue

    df = pd.read_csv(best_path)

    feature_cols = [c for c in df.columns if is_valid_feature(c)]

    total_feats = len(feature_cols)
    if total_feats == 0:
        print(f"[WARN] No feature columns found in: {best_path}")
        continue

    counts = {}
    for p in PREFIXES:
        counts[p] = sum(str(c).startswith(p) for c in feature_cols)

    image_count = sum(counts.values())
    tabular_count = total_feats - image_count

    # composition inside selected set
    femur_pct = 100 * counts["image_femur"] / total_feats
    head_pct = 100 * counts["image_head"] / total_feats
    abdomen_pct = 100 * counts["image_abdomen"] / total_feats
    image_pct = 100 * image_count / total_feats
    tabular_pct = 100 * tabular_count / total_feats

    # relative to total available
    total_feats_pct_all = 100 * total_feats / total_available if total_available else 0
    femur_pct_all = 100 * counts["image_femur"] / total_femur if total_femur else 0
    head_pct_all = 100 * counts["image_head"] / total_head if total_head else 0
    abdomen_pct_all = 100 * counts["image_abdomen"] / total_abdomen if total_abdomen else 0
    image_pct_all = 100 * image_count / total_image if total_image else 0
    tabular_pct_all = 100 * tabular_count / total_tabular if total_tabular else 0

    rows.append({
        "experiment": run,
        "total_features": total_feats,
        "total_features_%_of_all": round(total_feats_pct_all, 3),

        "femur_count": counts["image_femur"],
        "femur_%_within_selected": round(femur_pct, 3),
        "femur_%_of_all_femur": round(femur_pct_all, 3),

        "head_count": counts["image_head"],
        "head_%_within_selected": round(head_pct, 3),
        "head_%_of_all_head": round(head_pct_all, 3),

        "abdomen_count": counts["image_abdomen"],
        "abdomen_%_within_selected": round(abdomen_pct, 3),
        "abdomen_%_of_all_abdomen": round(abdomen_pct_all, 3),

        "image_total_count": image_count,
        "image_total_%_within_selected": round(image_pct, 3),
        "image_total_%_of_all_image": round(image_pct_all, 3),

        "tabular_count": tabular_count,
        "tabular_%_within_selected": round(tabular_pct, 3),
        "tabular_%_of_all_tabular": round(tabular_pct_all, 3),
    })

results = pd.DataFrame(rows).sort_values("experiment")
print(results)

mean_row = {"experiment": "MEAN"}
std_row = {"experiment": "STD"}

for col in results.columns:
    if col != "experiment":
        mean_row[col] = round(results[col].mean(), 3)
        std_row[col] = round(results[col].std(), 3)

results_out = pd.concat(
    [results, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)

out_csv = "../results/3planes_raw_features/exp3_mlp/3planes_feature_counts_and_percentages_per_experiment.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
results_out.to_csv(out_csv, index=False)

print(f"\nSaved → {out_csv}")

Total available features: 2396
  Femur:   768
  Head:    768
  Abdomen: 768
  Image total: 2304
  Tabular: 92
Found 30 experiments in: ../logs/MedViT2-nopt/3planes_raw_features/exp3_mlp
                        experiment  total_features  total_features_%_of_all  \
0   exp_2026-02-24 11:41:09.666690             792                   33.055   
1   exp_2026-02-24 13:55:33.266046            2396                  100.000   
2   exp_2026-02-24 18:20:27.568854            2396                  100.000   
3   exp_2026-02-24 20:35:57.995552            2389                   99.708   
4   exp_2026-02-25 00:24:12.175954             464                   19.366   
5   exp_2026-02-25 03:55:28.073646             257                   10.726   
6   exp_2026-02-25 07:16:39.428256            1428                   59.599   
7   exp_2026-02-25 10:37:28.241301            2299                   95.952   
8   exp_2026-02-25 17:04:03.965377             207                    8.639   
9   exp_2026-02-25 20:33